# 01: Data Exploration & Analysis

## Objectives
- Analyze Food-101 dataset structure and distribution
- Visualize sample images from each class
- Calculate comprehensive statistics (dimensions, formats, imbalances)
- Detect corrupted or problematic images
- Prepare data for MLflow tracking

## Dataset Information
- **Dataset**: Food-101 (101 food categories)
- **Images**: ~101,000 total images
- **Selected Classes**: 10 classes for this project
- **Image Format**: JPG
- **Typical Size**: Variable, will be standardized to 224x224

In [1]:
# Import libraries
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageStat
from pathlib import Path
import warnings
from collections import defaultdict, Counter
import random
from tqdm import tqdm
import json
from datetime import datetime

# MLflow imports
import mlflow
import mlflow.pytorch
from src.mlflow.tracker import MLflowTracker
from src.mlflow.config import DATASET_CONFIG

# Set random seeds for reproducibility
np.random.seed(42)
random.seed(42)

# Configuration
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")
warnings.filterwarnings('ignore')

# Paths
DATA_DIR = Path('./data')
FOOD101_DIR = DATA_DIR / 'food-101'
FOOD10_DIR = DATA_DIR / 'food10'
MLFLOW_DIR = Path('./mlruns')

print(f"Data directory: {DATA_DIR}")
print(f"Food-101 directory: {FOOD101_DIR}")
print(f"Food-10 directory: {FOOD10_DIR}")

c:\Users\dell\Downloads\project ai\project ai\venv\lib\site-packages\pydantic\_internal\_config.py:386: UserWarning: Valid config keys have changed in V2:
* 'schema_extra' has been renamed to 'json_schema_extra'
  warnings.warn(message, UserWarning)


ModuleNotFoundError: No module named 'src'

In [ ]:
# Selected classes for this project
SELECTED_CLASSES = [
    'pizza', 'hamburger', 'hot_dog', 'french_fries',
    'ice_cream', 'omelette', 'pancakes', 'ramen', 'steak', 'fried_rice'
]

print(f"Selected {len(SELECTED_CLASSES)} classes:")
for i, cls in enumerate(SELECTED_CLASSES, 1):
    print(f"  {i:2d}. {cls}")

# Initialize MLflow tracker
try:
    tracker = MLflowTracker(experiment_name="data-exploration")
    tracker.start_run(run_name="data_exploration_analysis")
    print("✅ MLflow tracking initialized")
except Exception as e:
    print(f"⚠️ MLflow tracking not available: {e}")
    tracker = None

## Step 1: Dataset Structure Analysis

In [ ]:
def analyze_dataset_structure():
    """Analyze the structure of Food-101 dataset"""
    
    print("🔍 Analyzing Food-101 Dataset Structure")
    print("=" * 50)
    
    if not FOOD101_DIR.exists():
        print(f"❌ Food-101 directory not found: {FOOD101_DIR}")
        print("Please run data preparation first.")
        return None
    
    # Check main directories
    images_dir = FOOD101_DIR / 'images'
    meta_dir = FOOD101_DIR / 'meta'
    
    print(f"📁 Images directory: {images_dir.exists()}")
    print(f"📁 Meta directory: {meta_dir.exists()}")
    
    # Analyze images directory
    if images_dir.exists():
        all_classes = sorted([d.name for d in images_dir.iterdir() if d.is_dir()])
        print(f"\n📊 Dataset Overview:")
        print(f"  Total classes: {len(all_classes)}")
        print(f"  First 10 classes: {all_classes[:10]}")
        
        # Count images per class
        class_counts = {}
        total_images = 0
        corrupted_count = 0
        
        print(f"\n📈 Counting images per class...")
        for class_name in tqdm(all_classes, desc="Analyzing classes"):
            class_dir = images_dir / class_name
            if class_dir.exists():
                images = list(class_dir.glob('*.jpg'))
                
                # Check for corrupted images
                valid_images = []
                for img_path in images:
                    try:
                        with Image.open(img_path) as img:
                            img.verify()
                        valid_images.append(img_path)
                    except Exception:
                        corrupted_count += 1
                
                class_counts[class_name] = len(valid_images)
                total_images += len(valid_images)
        
        print(f"\n📊 Image Statistics:")
        print(f"  Total valid images: {total_images:,}")
        print(f"  Corrupted images: {corrupted_count}")
        print(f"  Average images per class: {total_images/len(all_classes):.1f}")
        
        # Check our selected classes
        print(f"\n🎯 Selected Classes Analysis:")
        selected_counts = {}
        missing_classes = []
        
        for cls in SELECTED_CLASSES:
            if cls in class_counts:
                selected_counts[cls] = class_counts[cls]
                print(f"  ✅ {cls:15s}: {class_counts[cls]:4d} images")
            else:
                missing_classes.append(cls)
                print(f"  ❌ {cls:15s}: NOT FOUND")
        
        if missing_classes:
            print(f"\n⚠️ Missing classes: {missing_classes}")
        
        return {
            'all_classes': all_classes,
            'class_counts': class_counts,
            'selected_counts': selected_counts,
            'missing_classes': missing_classes,
            'total_images': total_images,
            'corrupted_images': corrupted_count
        }
    
    else:
        print("❌ Images directory not found!")
        return None

# Run analysis
dataset_info = analyze_dataset_structure()

# Log to MLflow
if tracker and dataset_info:
    tracker.log_params({
        'total_classes': len(dataset_info['all_classes']),
        'total_images': dataset_info['total_images'],
        'corrupted_images': dataset_info['corrupted_images'],
        'selected_classes': len(SELECTED_CLASSES),
        'missing_selected_classes': len(dataset_info['missing_classes'])
    }, prefix='dataset_structure')

## Step 2: Class Distribution Analysis

In [ ]:
def analyze_class_distribution(dataset_info):
    """Analyze and visualize class distribution"""
    
    if not dataset_info:
        print("❌ No dataset info available")
        return
    
    print("\n📊 Class Distribution Analysis")
    print("=" * 40)
    
    class_counts = dataset_info['class_counts']
    
    # Create visualizations
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle('Food-101 Dataset Distribution Analysis', fontsize=16, fontweight='bold')
    
    # 1. Overall class distribution
    classes = list(class_counts.keys())
    counts = list(class_counts.values())
    
    axes[0,0].bar(range(len(counts)), counts)
    axes[0,0].set_title('Overall Class Distribution')
    axes[0,0].set_xlabel('Class Index')
    axes[0,0].set_ylabel('Number of Images')
    axes[0,0].grid(True, alpha=0.3)
    
    # 2. Class distribution histogram
    axes[0,1].hist(counts, bins=20, alpha=0.7, color='skyblue', edgecolor='black')
    axes[0,1].set_title('Distribution of Image Counts per Class')
    axes[0,1].set_xlabel('Number of Images')
    axes[0,1].set_ylabel('Number of Classes')
    axes[0,1].grid(True, alpha=0.3)
    
    # 3. Top 20 classes by image count
    sorted_counts = sorted(class_counts.items(), key=lambda x: x[1], reverse=True)[:20]
    top_classes = [x[0] for x in sorted_counts]
    top_counts = [x[1] for x in sorted_counts]
    
    axes[1,0].barh(range(len(top_counts)), top_counts)
    axes[1,0].set_yticks(range(len(top_classes)))
    axes[1,0].set_yticklabels(top_classes)
    axes[1,0].set_title('Top 20 Classes by Image Count')
    axes[1,0].set_xlabel('Number of Images')
    axes[1,0].grid(True, alpha=0.3)
    
    # 4. Selected classes comparison
    if dataset_info['selected_counts']:
        selected_names = list(dataset_info['selected_counts'].keys())
        selected_counts = list(dataset_info['selected_counts'].values())
        
        colors = ['green' if count >= 800 else 'orange' if count >= 500 else 'red' 
                for count in selected_counts]
        
        bars = axes[1,1].bar(selected_names, selected_counts, color=colors, alpha=0.7)
        axes[1,1].set_title('Selected Classes Image Count')
        axes[1,1].set_xlabel('Class')
        axes[1,1].set_ylabel('Number of Images')
        axes[1,1].tick_params(axis='x', rotation=45)
        axes[1,1].grid(True, alpha=0.3)
        
        # Add value labels on bars
        for bar, count in zip(bars, selected_counts):
            height = bar.get_height()
            axes[1,1].text(bar.get_x() + bar.get_width()/2., height + 5,
                          f'{count}', ha='center', va='bottom')
    
    plt.tight_layout()
    plt.show()
    
    # Calculate statistics
    mean_count = np.mean(counts)
    std_count = np.std(counts)
    min_count = np.min(counts)
    max_count = np.max(counts)
    
    print(f"\n📈 Distribution Statistics:")
    print(f"  Mean images per class: {mean_count:.1f}")
    print(f"  Std deviation: {std_count:.1f}")
    print(f"  Min images: {min_count}")
    print(f"  Max images: {max_count}")
    print(f"  Coefficient of variation: {std_count/mean_count:.3f}")
    
    # Log to MLflow
    if tracker:
        tracker.log_metrics({
            'mean_images_per_class': mean_count,
            'std_images_per_class': std_count,
            'min_images_per_class': min_count,
            'max_images_per_class': max_count,
            'cv_images_per_class': std_count/mean_count
        }, prefix='class_distribution')

# Run analysis
if dataset_info:
    analyze_class_distribution(dataset_info)

## Step 3: Image Properties Analysis

In [ ]:
def analyze_image_properties(dataset_info, sample_size=100):
    """Analyze image properties like dimensions, formats, sizes"""
    
    if not dataset_info or not dataset_info['selected_counts']:
        print("❌ No selected classes available")
        return
    
    print("\n🔍 Image Properties Analysis")
    print("=" * 40)
    
    images_dir = FOOD101_DIR / 'images'
    selected_classes = list(dataset_info['selected_counts'].keys())
    
    # Collect image properties
    image_properties = {
        'widths': [],
        'heights': [],
        'aspect_ratios': [],
        'file_sizes': [],
        'formats': [],
        'class_names': [],
        'file_paths': []
    }
    
    print(f"📸 Analyzing {sample_size} images...")
    
    # Sample images from each class
    for class_name in tqdm(selected_classes, desc="Analyzing images"):
        class_dir = images_dir / class_name
        if not class_dir.exists():
            continue
        
        images = list(class_dir.glob('*.jpg'))
        if not images:
            continue
        
        # Sample images from this class
        n_samples = min(len(images), sample_size // len(selected_classes))
        sampled_images = random.sample(images, n_samples)
        
        for img_path in sampled_images:
            try:
                # Get file size
                file_size = img_path.stat().st_size / 1024  # KB
                
                # Get image dimensions
                with Image.open(img_path) as img:
                    width, height = img.size
                    aspect_ratio = width / height
                    
                    image_properties['widths'].append(width)
                    image_properties['heights'].append(height)
                    image_properties['aspect_ratios'].append(aspect_ratio)
                    image_properties['file_sizes'].append(file_size)
                    image_properties['formats'].append(img.format)
                    image_properties['class_names'].append(class_name)
                    image_properties['file_paths'].append(str(img_path))
                    
            except Exception as e:
                print(f"⚠️ Error analyzing {img_path}: {e}")
                continue
    
    print(f"\n✅ Analyzed {len(image_properties['widths'])} images")
    
    # Create visualizations
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    fig.suptitle('Image Properties Analysis', fontsize=16, fontweight='bold')
    
    # 1. Width distribution
    axes[0,0].hist(image_properties['widths'], bins=30, alpha=0.7, color='blue', edgecolor='black')
    axes[0,0].set_title('Image Width Distribution')
    axes[0,0].set_xlabel('Width (pixels)')
    axes[0,0].set_ylabel('Count')
    axes[0,0].grid(True, alpha=0.3)
    
    # 2. Height distribution
    axes[0,1].hist(image_properties['heights'], bins=30, alpha=0.7, color='green', edgecolor='black')
    axes[0,1].set_title('Image Height Distribution')
    axes[0,1].set_xlabel('Height (pixels)')
    axes[0,1].set_ylabel('Count')
    axes[0,1].grid(True, alpha=0.3)
    
    # 3. Aspect ratio distribution
    axes[0,2].hist(image_properties['aspect_ratios'], bins=30, alpha=0.7, color='orange', edgecolor='black')
    axes[0,2].set_title('Aspect Ratio Distribution')
    axes[0,2].set_xlabel('Aspect Ratio (W/H)')
    axes[0,2].set_ylabel('Count')
    axes[0,2].grid(True, alpha=0.3)
    
    # 4. File size distribution
    axes[1,0].hist(image_properties['file_sizes'], bins=30, alpha=0.7, color='red', edgecolor='black')
    axes[1,0].set_title('File Size Distribution')
    axes[1,0].set_xlabel('File Size (KB)')
    axes[1,0].set_ylabel('Count')
    axes[1,0].grid(True, alpha=0.3)
    
    # 5. Width vs Height scatter
    axes[1,1].scatter(image_properties['widths'], image_properties['heights'], alpha=0.5, s=10)
    axes[1,1].set_title('Width vs Height')
    axes[1,1].set_xlabel('Width (pixels)')
    axes[1,1].set_ylabel('Height (pixels)')
    axes[1,1].grid(True, alpha=0.3)
    
    # 6. Aspect ratio vs file size
    axes[1,2].scatter(image_properties['aspect_ratios'], image_properties['file_sizes'], alpha=0.5, s=10)
    axes[1,2].set_title('Aspect Ratio vs File Size')
    axes[1,2].set_xlabel('Aspect Ratio')
    axes[1,2].set_ylabel('File Size (KB)')
    axes[1,2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Calculate statistics
    stats = {
        'mean_width': np.mean(image_properties['widths']),
        'mean_height': np.mean(image_properties['heights']),
        'mean_aspect_ratio': np.mean(image_properties['aspect_ratios']),
        'mean_file_size': np.mean(image_properties['file_sizes']),
        'std_width': np.std(image_properties['widths']),
        'std_height': np.std(image_properties['heights']),
        'min_width': np.min(image_properties['widths']),
        'max_width': np.max(image_properties['widths']),
        'min_height': np.min(image_properties['heights']),
        'max_height': np.max(image_properties['heights']),
        'total_samples': len(image_properties['widths'])
    }
    
    print(f"\n📊 Image Properties Statistics:")
    print(f"  Samples analyzed: {stats['total_samples']}")
    print(f"  Mean dimensions: {stats['mean_width']:.1f} x {stats['mean_height']:.1f}")
    print(f"  Width range: {stats['min_width']} - {stats['max_width']}")
    print(f"  Height range: {stats['min_height']} - {stats['max_height']}")
    print(f"  Mean aspect ratio: {stats['mean_aspect_ratio']:.3f}")
    print(f"  Mean file size: {stats['mean_file_size']:.1f} KB")
    
    # Recommendations for preprocessing
    print(f"\n💡 Preprocessing Recommendations:")
    if stats['mean_aspect_ratio'] > 1.2:
        print(f"  • Images are generally wider than tall")
    elif stats['mean_aspect_ratio'] < 0.8:
        print(f"  • Images are generally taller than wide")
    else:
        print(f"  • Images are roughly square")
    
    target_size = 224
    print(f"  • Recommended resize: {target_size}x{target_size} (standard for ImageNet models)")
    print(f"  • Consider center crop to maintain aspect ratio")
    
    # Log to MLflow
    if tracker:
        tracker.log_params(stats, prefix='image_properties')
    
    return image_properties, stats

# Run analysis
image_props, img_stats = analyze_image_properties(dataset_info)

## Step 4: Sample Images Visualization

In [ ]:
def visualize_sample_images(dataset_info, samples_per_class=3):
    """Visualize sample images from each class"""
    
    if not dataset_info or not dataset_info['selected_counts']:
        print("❌ No selected classes available")
        return
    
    print("\n🖼️ Sample Images Visualization")
    print("=" * 40)
    
    images_dir = FOOD101_DIR / 'images'
    selected_classes = list(dataset_info['selected_counts'].keys())
    
    # Create grid layout
    n_classes = len(selected_classes)
    fig, axes = plt.subplots(n_classes, samples_per_class, figsize=(15, 3*n_classes))
    fig.suptitle('Sample Images from Selected Classes', fontsize=16, fontweight='bold')
    
    if n_classes == 1:
        axes = axes.reshape(1, -1)
    
    for i, class_name in enumerate(selected_classes):
        class_dir = images_dir / class_name
        if not class_dir.exists():
            continue
        
        images = list(class_dir.glob('*.jpg'))
        if not images:
            continue
        
        # Sample random images
        sampled_images = random.sample(images, min(samples_per_class, len(images)))
        
        for j, img_path in enumerate(sampled_images):
            try:
                with Image.open(img_path) as img:
                    axes[i, j].imshow(img)
                    axes[i, j].set_title(f'{class_name}', fontsize=10)
                    axes[i, j].axis('off')
                    
                    # Add image info
                    width, height = img.size
                    file_size = img_path.stat().st_size / 1024  # KB
                    axes[i, j].text(0.02, 0.98, f'{width}x{height}\n{file_size:.1f}KB', 
                                   transform=axes[i, j].transAxes, fontsize=8,
                                   verticalalignment='top',
                                   bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
                    
            except Exception as e:
                axes[i, j].text(0.5, 0.5, f'Error loading image', 
                               transform=axes[i, j].transAxes, ha='center', va='center')
                axes[i, j].axis('off')
        
        # Add class label on the left
        axes[i, 0].set_ylabel(f'{class_name}\n({dataset_info["selected_counts"][class_name]} images)', 
                           rotation=0, ha='right', va='center', fontsize=12, fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    # Save visualization
    plt.savefig('sample_images_grid.png', dpi=300, bbox_inches='tight')
    print(f"✅ Sample images visualization saved")
    
    # Log to MLflow
    if tracker:
        tracker.log_artifact('sample_images_grid.png', 'visualizations')

# Run visualization
visualize_sample_images(dataset_info)

## Step 5: Data Quality Assessment

In [ ]:
def assess_data_quality(dataset_info, sample_size=50):
    """Assess data quality and detect problematic images"""
    
    if not dataset_info or not dataset_info['selected_counts']:
        print("❌ No selected classes available")
        return
    
    print("\n🔍 Data Quality Assessment")
    print("=" * 40)
    
    images_dir = FOOD101_DIR / 'images'
    selected_classes = list(dataset_info['selected_counts'].keys())
    
    quality_issues = {
        'corrupted': [],
        'very_small': [],
        'very_large': [],
        'extreme_aspect_ratio': [],
        'grayscale': [],
        'low_contrast': [],
        'duplicate_content': []
    }
    
    print(f"🔍 Analyzing {sample_size} images per class for quality issues...")
    
    for class_name in tqdm(selected_classes, desc="Quality assessment"):
        class_dir = images_dir / class_name
        if not class_dir.exists():
            continue
        
        images = list(class_dir.glob('*.jpg'))
        if not images:
            continue
        
        # Sample images
        n_samples = min(len(images), sample_size)
        sampled_images = random.sample(images, n_samples)
        
        for img_path in sampled_images:
            try:
                with Image.open(img_path) as img:
                    # Check for corruption
                    try:
                        img.verify()
                    except Exception:
                        quality_issues['corrupted'].append(str(img_path))
                        continue
                    
                    # Reload after verify
                    img = Image.open(img_path)
                    
                    # Check dimensions
                    width, height = img.size
                    aspect_ratio = width / height
                    
                    if width < 100 or height < 100:
                        quality_issues['very_small'].append(str(img_path))
                    elif width > 1000 or height > 1000:
                        quality_issues['very_large'].append(str(img_path))
                    
                    if aspect_ratio < 0.5 or aspect_ratio > 2.0:
                        quality_issues['extreme_aspect_ratio'].append(str(img_path))
                    
                    # Check if grayscale
                    if img.mode != 'RGB':
                        quality_issues['grayscale'].append(str(img_path))
                    
                    # Check contrast
                    img_gray = img.convert('L')
                    hist = img_gray.histogram()
                    contrast = np.std(hist) / np.mean(hist) if np.mean(hist) > 0 else 0
                    
                    if contrast < 0.1:
                        quality_issues['low_contrast'].append(str(img_path))
                    
            except Exception as e:
                quality_issues['corrupted'].append(str(img_path))
    
    # Print quality assessment results
    print(f"\n📊 Quality Assessment Results:")
    
    total_issues = sum(len(issues) for issues in quality_issues.values())
    print(f"  Total issues found: {total_issues}")
    
    for issue_type, issues in quality_issues.items():
        if issues:
            print(f"  {issue_type.replace('_', ' ').title()}: {len(issues)} images")
    
    # Create quality report
    quality_report = {
        'assessment_date': datetime.now().isoformat(),
        'sample_size_per_class': sample_size,
        'total_samples': sample_size * len(selected_classes),
        'quality_issues': {k: len(v) for k, v in quality_issues.items()},
        'total_issues': total_issues
    }
    
    # Visualize quality issues
    if total_issues > 0:
        fig, axes = plt.subplots(2, 2, figsize=(15, 10))
        fig.suptitle('Data Quality Issues Distribution', fontsize=16, fontweight='bold')
        
        issue_types = list(quality_issues.keys())
        issue_counts = [len(quality_issues[issue]) for issue in issue_types]
        
        # Bar chart of issues
        axes[0,0].bar(issue_types, issue_counts)
        axes[0,0].set_title('Quality Issues by Type')
        axes[0,0].set_ylabel('Number of Images')
        axes[0,0].tick_params(axis='x', rotation=45)
        axes[0,0].grid(True, alpha=0.3)
        
        # Pie chart
        if total_issues > 0:
            axes[0,1].pie(issue_counts, labels=issue_types, autopct='%1.1f%%')
            axes[0,1].set_title('Quality Issues Distribution')
        
        # Recommendations
        recommendations = []
        if quality_issues['corrupted']:
            recommendations.append('Remove corrupted images')
        if quality_issues['very_small']:
            recommendations.append('Consider upscaling small images')
        if quality_issues['very_large']:
            recommendations.append('Consider downscaling large images')
        if quality_issues['extreme_aspect_ratio']:
            recommendations.append('Handle extreme aspect ratios carefully')
        if quality_issues['grayscale']:
            recommendations.append('Convert grayscale to RGB or handle separately')
        if quality_issues['low_contrast']:
            recommendations.append('Apply contrast enhancement')
        
        axes[1,0].text(0.1, 0.9, 'Recommendations:', transform=axes[1,0].transAxes, fontsize=12, fontweight='bold')
        for i, rec in enumerate(recommendations):
            axes[1,0].text(0.1, 0.8 - i*0.1, f'• {rec}', transform=axes[1,0].transAxes, fontsize=10)
        axes[1,0].axis('off')
        
        # Hide unused subplots
        axes[1,1].axis('off')
        
        plt.tight_layout()
        plt.show()
        
        # Save quality report
        plt.savefig('data_quality_assessment.png', dpi=300, bbox_inches='tight')
        
        # Log to MLflow
        if tracker:
            tracker.log_params(quality_report['quality_issues'], prefix='data_quality')
            tracker.log_artifact('data_quality_assessment.png', 'quality_analysis')
            
            # Save detailed report
            with open('quality_report.json', 'w') as f:
                json.dump(quality_report, f, indent=2)
            tracker.log_artifact('quality_report.json', 'quality_analysis')
    
    else:
        print("  ✅ No significant quality issues detected!")
    
    return quality_issues, quality_report

# Run quality assessment
quality_issues, quality_report = assess_data_quality(dataset_info)

## Step 6: Summary and MLflow Logging

In [ ]:
def create_data_summary(dataset_info, image_stats, quality_report):
    """Create comprehensive data summary and log to MLflow"""
    
    print("\n📋 Data Exploration Summary")
    print("=" * 50)
    
    if not dataset_info:
        print("❌ No dataset information available")
        return
    
    # Create summary dictionary
    summary = {
        'dataset_name': 'Food-101',
        'analysis_date': datetime.now().isoformat(),
        'total_classes': len(dataset_info['all_classes']),
        'selected_classes': len(SELECTED_CLASSES),
        'total_images': dataset_info['total_images'],
        'corrupted_images': dataset_info['corrupted_images'],
        'missing_selected_classes': len(dataset_info['missing_classes']),
    }
    
    if image_stats:
        summary.update({
            'mean_image_width': image_stats['mean_width'],
            'mean_image_height': image_stats['mean_height'],
            'mean_aspect_ratio': image_stats['mean_aspect_ratio'],
            'mean_file_size_kb': image_stats['mean_file_size'],
        })
    
    if quality_report:
        summary.update({
            'quality_issues_total': quality_report['total_issues'],
            'quality_sample_size': quality_report['sample_size_per_class'],
        })
    
    # Print summary
    print(f"📊 Dataset: {summary['dataset_name']}")
    print(f"📈 Total Classes: {summary['total_classes']}")
    print(f"🎯 Selected Classes: {summary['selected_classes']}")
    print(f"🖼️ Total Images: {summary['total_images']:,}")
    print(f"⚠️ Corrupted Images: {summary['corrupted_images']}")
    
    if image_stats:
        print(f"📏 Mean Image Size: {summary['mean_image_width']:.0f}x{summary['mean_image_height']:.0f}")
        print(f"📐 Mean Aspect Ratio: {summary['mean_aspect_ratio']:.3f}")
        print(f"💾 Mean File Size: {summary['mean_file_size_kb']:.1f} KB")
    
    if quality_report:
        print(f"🔍 Quality Issues: {summary['quality_issues_total']}")
    
    # Recommendations
    print(f"\n💡 Recommendations for Model Training:")
    
    if summary['missing_selected_classes'] > 0:
        print(f"  ⚠️ Replace missing classes: {dataset_info['missing_classes']}")
    
    if image_stats and image_stats['mean_aspect_ratio'] > 1.2:
        print(f"  📐 Images are wider than tall - consider center cropping")
    elif image_stats and image_stats['mean_aspect_ratio'] < 0.8:
        print(f"  📐 Images are taller than wide - consider center cropping")
    
    if quality_report and quality_report['total_issues'] > 0:
        print(f"  🔍 Address {quality_report['total_issues']} quality issues found")
    
    print(f"  📏 Standardize images to 224x224 for ImageNet models")
    print(f"  🎯 Use data augmentation to improve model robustness")
    print(f"  📊 Implement proper train/val/test splits")
    
    # Log to MLflow
    if tracker:
        tracker.log_params(summary, prefix='data_summary')
        
        # Log this notebook as artifact
        tracker.log_artifact('01_data_exploration_fixed.ipynb', 'notebooks')
        
        # Final summary metrics
        tracker.log_metrics({
            'data_quality_score': max(0, 100 - (quality_report['total_issues'] / quality_report['total_samples'] * 100) if quality_report['total_samples'] > 0 else 100),
            'dataset_completeness': (summary['selected_classes'] - summary['missing_selected_classes']) / summary['selected_classes'] * 100
        })
        
        print(f"\n✅ Data exploration logged to MLflow successfully!")
    
    return summary

# Create summary
data_summary = create_data_summary(dataset_info, img_stats if 'img_stats' in locals() else None, quality_report if 'quality_report' in locals() else None)

# End MLflow run
if tracker:
    tracker.end_run()
    print(f"\n🎉 Data exploration completed and logged to MLflow!")
    print(f"📊 View results: http://localhost:5000")